# Building Agents with LangGraph Series #7: Building a Multi-Agent System LangGraph

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/building-agents-with-langgraph-course-7-building-a-multi-agent-system-langgraph). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/series/langgraph/LangGraph_Series_7_Multi_Agent_System.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/series/langgraph/LangGraph_Series_7_Multi_Agent_System.ipynb)


## Before you begin

Add provider keys through Colab secrets or environment variables. Run cells in order; the multi-agent graph depends on definitions created earlier in the lesson.

This notebook intentionally contains no saved execution outputs or credentials.


## Environment setup


In [ ]:
!pip install -q langgraph langchain-openai tavily-python python-dotenv


Welcome to the seventh article in our ongoing “Building Agents with LangGraph” series! So far, we’ve explored agents with a single Large Language Model (LLM) and a simple state. Now, we’re going to create a much more sophisticated agent composed of multiple, distinct LLM calls and a more complex state.

In this tutorial, we will build a multi-agent system that collaborates to write an essay. This system will follow a cyclical, reflective process: it will plan, research, write, and then critique its own work to produce a refined final draft.

This demonstrates how LangGraph can be used to orchestrate complex, stateful workflows involving multiple specialized “agents” or roles.

## Table of Contents

This article is the Seventh Article in the ongoing series of Building LLM Agents with LangGraph:

- Introduction to Agents & LangGraph (Published!)
- Building Simple ReAct Agent from Scratch (Published!)
- Main Building Units of LangGraph (Published!)
- Agentic Search Tools in LangGraph (Published!)
- Persistence and Streaming in LangGraph (Published!)
- Human in the Loop in LLM Agents (Published!)
- Putting it All Together! Multi-Agent System LangGraph (You are here!)

This series is designed to take readers from foundational knowledge to advanced practices in building LLM agents with LangGraph.

Each article delves into essential components, such as constructing simple ReAct agents from scratch, leveraging LangGraph’s building units, utilizing agentic search tools, implementing persistence and streaming capabilities, integrating human-in-the-loop interactions, and culminating in the creation of a fully functional essay-writing agent.

By the end of this series, you will have a comprehensive understanding of LangGraph, practical skills to design and deploy LLM agents, and the confidence to build customized AI-driven workflows tailored to diverse applications.

---

## 1. Project Setup and Dependencies

First, let’s set up our environment. We’ll load our API keys from a .env file and import the necessary libraries. This includes components from langgraph for building the graph, langchain_core for message types, and langchain_openai for the model. We’ll also set up an in-memory checkpointer using SqliteSaver to persist the state of our graph.


In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

memory = SqliteSaver.from_conn_string(":memory:")


---

## 2. Defining the Agent’s State

The state is the backbone of our agent, defining the structure of data that gets passed between nodes. For our essay writer, the state needs to track several pieces of information: the initial task, the plan, the research content, the draft, critiques, and the number of revisions.

We define this using a TypedDict:


In [ ]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int


Next, we initialize our model. We will use GPT-4 for this example.


In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4", temperature=0)


---

## 3. Creating the Agent Roles (Nodes)

Our multi-agent system will consist of several specialized roles, each represented by a node in our graph. Each role has a specific prompt that guides the LLM to perform its task.

### 3.1. The Planner Agent

The first step is to create a high-level plan. The planner node takes the user’s task and generates an essay outline.


In [ ]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. \
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes \
or instructions for the sections."""


### 3.2. The Writer Agent

The writer node is responsible for generating the essay draft based on the plan and any research content. It’s also designed to revise its draft based on critiques.


In [ ]:
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.\
Generate the best essay possible for the user's request and the initial outline. \
If the user provides critique, respond with a revised version of your previous attempts. \
Utilize all the information below as needed:

------

{content}"""


### 3.3. The Reflector Agent

The `reflection` node acts as a critic. It reviews the draft and provides constructive feedback and recommendations for improvement.


In [ ]:
REFLECTION_PROMPT = """You are a teacher grading an essay submission. \
Generate critique and recommendations for the user's submission. \
Provide detailed recommendations, including requests for length, depth, style, etc."""


### 3.4. The Researcher & Critique Agent

To ensure the essay is well-informed, we have two research nodes that generate search queries. One works from the initial plan, and the other works from the critique to find information for revisions. We’ll use the Tavily search API to execute these queries.


In [ ]:
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can \
be used when writing the following essay. Generate a list of search queries that will gather \
any relevant information. Only generate 3 queries max."""


In [ ]:
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can \
be used when making any requested revisions (as outlined below). \
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


To handle the output of the research query generator, we define a Pydantic model.


In [ ]:
from langchain_core.pydantic_v1 import BaseModel

class Queries(BaseModel):
    queries: List[str]


And we initialize our Tavily search client.


In [ ]:
from tavily import TavilyClient
import os
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


---

## 4. Implementing the Node Functions

With our prompts defined, we now create the Python functions that will execute the logic for each node. These functions take the current AgentState as input and return a dictionary with the updated state values.


In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
        ]
    response = model.invoke(messages)
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1
    }

def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}


---

## 5. Defining the Graph’s Logic and Edges

The power of LangGraph lies in its ability to define cyclical and conditional workflows. Our essay writer needs a loop for revisions. We’ll create a function, should_continue, to decide whether to proceed with another revision or to end the process.


In [ ]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"


Now, we can wire everything together. We’ll instantiate a StateGraph, add our nodes, and define the edges that control the flow of execution.


In [ ]:
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

# Define edges
builder.set_entry_point("planner")

builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")

builder.add_conditional_edges(
    "generate",
    should_continue,
    {END: END, "reflect": "reflect"}
)

builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")


Finally, we compile the graph, connecting it to our memory checkpointer.


In [ ]:
graph = builder.compile(checkpointer=memory)


---

## 6. Visualizing and Running the Agent

One of the great features of LangGraph is the ability to visualize the structure of your agent. This helps in debugging and understanding the flow.


In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_png())


![LangGraph multi-agent essay workflow from planner through research, generation, reflection, and completion](https://todatabeyond.com/articles/langgraph-course-7-workflow.png)

*The LangGraph essay-writing workflow cycles through planning, research, generation, reflection, and critique-driven research before completion.*

Now, let’s run our agent and see it in action. We’ll ask it to write an essay on the difference between AI and Generative AI, allowing for one revision (max_revisions: 2).


In [ ]:
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    'task': "what is the difference between AI and Generative AI?",
    "max_revisions": 2,
    "revision_number": 1,
}, thread):
    print(s)


**Expected output**

```text
{‘planner’: {‘plan’: ‘I. Introduction\n A. Definition of AI\n B. Definition of Generative AI\n C. Brief overview of the differences between AI and Generative AI\n\nII. Understanding Artificial Intelligence (AI)\n A. Explanation of AI\n B. Applications of AI\n C. Examples of AI technologies\n\nIII. Exploring Generative AI\n A. Definition of Generative AI\n B. How Generative AI differs from traditional AI\n C. Applications of Generative AI\n\nIV. Key Differences Between AI and Generative AI\n A. Data processing capabilities\n B. Creativity and innovation\n C. Learning and adaptation\n\nV. Real-World Examples\n A. AI applications in various industries\n B. Generative AI applications and their impact\n\nVI. Future Implications\n A. Potential advancements in AI and Generative AI\n B. Ethical considerations and challenges\n\nVII. Conclusion\n A. Recap of the main differences between AI and Generative AI\n B. Final thoughts on the future of AI and Generative AI\n\nNotes:\n- Provide clear definitions and examples to help differentiate between AI and Generative AI.\n- Include real-world applications to illustrate how each technology is used in various industries.\n- Discuss the potential future advancements and ethical considerations related to AI and Generative AI.’}}
{‘research_plan’: {‘content’: [‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’, ‘Generative AI is an AI that can create new content, such as images, video and text, based on the data on which it is trained. In other words, generative AI creates new content based on patterns in data and traditional AI focuses on analyzing and classifying existing data. For example, traditional AI could analyze user behavior data first, then generative AI could work to create highly personalized content from the data. This is our AI-powered platform that combines orchestration, automation and other technologies like traditional AI and generative AI to help you create smarter business operations. They added generative AI and automation to their processes and are now responding 60% faster to customer inquiries.’, ‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’, ‘# The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone Traditional AI systems are primarily used to analyze data and make predictions, while generative AI goes a step further by creating new data similar to its training data. Traditional AI can analyze data and tell you what it sees, but generative AI can use that same data to create something entirely new. For instance, a traditional AI could analyze user behavior data, and a generative AI could use this analysis to create personalized content.’, ‘Generative AI is an AI that can create new content, such as images, video and text, based on the data on which it is trained. In other words, generative AI creates new content based on patterns in data and traditional AI focuses on analyzing and classifying existing data. For example, traditional AI could analyze user behavior data first, then generative AI could work to create highly personalized content from the data. This is our AI-powered platform that combines orchestration, automation and other technologies like traditional AI and generative AI to help you create smarter business operations. They added generative AI and automation to their processes and are now responding 60% faster to customer inquiries.’, ‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’]}}
{‘generate’: {‘draft’: “**Title: Exploring the Divide: Artificial Intelligence (AI) vs. Generative AI**\n\nI. Introduction\nArtificial Intelligence (AI) and Generative AI represent two distinct realms within the field of technology. While AI focuses on data analysis and task automation, Generative AI takes a leap forward by creating entirely new content based on learned patterns. This essay delves into the disparities between AI and Generative AI, shedding light on their unique characteristics and applications.\n\nII. Understanding Artificial Intelligence (AI)\nAI encompasses systems designed to interpret data and make informed predictions. From chatbots to recommendation engines, AI technologies are prevalent in various industries, streamlining processes and enhancing user experiences through data analysis and pattern recognition.\n\nIII. Exploring Generative AI\nGenerative AI, a subset of AI, is engineered to produce original and creative content. Unlike traditional AI, Generative AI is not confined to predefined tasks, allowing for interactive and collaborative content creation. Its applications span from generating text to crafting images, opening new avenues for innovation.\n\nIV. Key Differences Between AI and Generative AI\n1. Data Processing Capabilities: AI excels in processing existing data, while Generative AI leverages learned patterns to generate novel content.\n2. Creativity and Innovation: Generative AI introduces a revolutionary twist by fostering creativity and innovation through content creation.\n3. Learning and Adaptation: While AI learns from data to make predictions, Generative AI adapts this knowledge to generate fresh content, showcasing a higher level of adaptability.\n\nV. Real-World Examples\nAI finds applications in diverse industries, from healthcare to finance, optimizing operations and decision-making processes. On the other hand, Generative AI’s impact is evident in creative fields like art and design, where it aids in producing unique content and pushing boundaries of creativity.\n\nVI. Future Implications\nAs AI and Generative AI continue to evolve, advancements in technology hold promise for enhanced capabilities and efficiency. However, ethical considerations surrounding data privacy, bias, and job displacement remain critical areas of concern that necessitate careful navigation in the future landscape of AI technologies.\n\nVII. Conclusion\nIn conclusion, the distinction between AI and Generative AI lies in their core functionalities and applications. While AI focuses on data analysis and automation, Generative AI stands out for its creative content generation capabilities. Understanding these disparities and their implications is crucial for navigating the ever-evolving landscape of artificial intelligence technologies.”, ‘revision_number’: 2}}
{‘reflect’: {‘critique’: ‘Overall, your essay provides a clear and concise overview of the differences between Artificial Intelligence (AI) and Generative AI. You have effectively highlighted the key characteristics and applications of both technologies. However, there are a few areas where you can enhance your essay:\n\n1. **Depth and Analysis**: While you have outlined the basic differences between AI and Generative AI, consider delving deeper into specific examples or case studies to illustrate these disparities further. Providing real-world examples or discussing specific applications can help reinforce your points and make your essay more engaging.\n\n2. **Expansion on Future Implications**: In the section on future implications, you briefly touch on ethical considerations. Consider expanding on this aspect by discussing potential challenges and opportunities that may arise as AI and Generative AI technologies continue to advance. Addressing issues such as bias in algorithms, data privacy concerns, and the impact on the job market can add depth to your analysis.\n\n3. **Engagement with Counterarguments**: To strengthen your essay, consider addressing potential counterarguments or criticisms of AI and Generative AI technologies. By acknowledging differing perspectives and providing a balanced view, you can demonstrate a more comprehensive understanding of the topic.\n\n4. **Conclusion**: Your conclusion effectively summarizes the key points discussed in the essay. Consider adding a call to action or a thought-provoking statement that encourages readers to reflect on the implications of AI technologies in society.\n\n5. **Length and Detail**: While your essay is well-structured, consider expanding on each section to provide more detailed explanations and examples. This will help enrich your analysis and provide a more comprehensive understanding of the topic.\n\nOverall, your essay is a solid introduction to the topic of AI and Generative AI. By incorporating more depth, analysis, and real-world examples, you can elevate your essay to a more sophisticated level of discussion. Keep refining your writing by incorporating these suggestions to enhance the overall quality of your work.’}}
{‘research_critique’: {‘content’: [‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’, ‘Generative AI is an AI that can create new content, such as images, video and text, based on the data on which it is trained. In other words, generative AI creates new content based on patterns in data and traditional AI focuses on analyzing and classifying existing data. For example, traditional AI could analyze user behavior data first, then generative AI could work to create highly personalized content from the data. This is our AI-powered platform that combines orchestration, automation and other technologies like traditional AI and generative AI to help you create smarter business operations. They added generative AI and automation to their processes and are now responding 60% faster to customer inquiries.’, ‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’, ‘# The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone The Difference Between Generative AI And Traditional AI: An Easy Explanation For Anyone Traditional AI systems are primarily used to analyze data and make predictions, while generative AI goes a step further by creating new data similar to its training data. Traditional AI can analyze data and tell you what it sees, but generative AI can use that same data to create something entirely new. For instance, a traditional AI could analyze user behavior data, and a generative AI could use this analysis to create personalized content.’, ‘Generative AI is an AI that can create new content, such as images, video and text, based on the data on which it is trained. In other words, generative AI creates new content based on patterns in data and traditional AI focuses on analyzing and classifying existing data. For example, traditional AI could analyze user behavior data first, then generative AI could work to create highly personalized content from the data. This is our AI-powered platform that combines orchestration, automation and other technologies like traditional AI and generative AI to help you create smarter business operations. They added generative AI and automation to their processes and are now responding 60% faster to customer inquiries.’, ‘## What Is Generative AI? This allows generative AI systems to create original, creative content. On the other hand, generative AI systems are not limited to specific tasks. Generative AI systems, on the other hand, have unique use cases that revolve around content creation. Generative AI systems are more interactive and collaborative. While traditional AI excels at analyzing data and automating tasks, *generative AI adds a revolutionary twist by creating entirely new content*. **Generative AI:** a subset of AI that focuses on generating new content, such as text or images, based on patterns learned from data. While traditional AI excels at analyzing data and automating tasks, generative AI adds a revolutionary twist by creating entirely new content.’, “10 Powerful Examples Of AI Applications In Today’s World · Automated customer support · Personalized shopping experience · Healthcare · Finance · Smart cars and”, “* ***Capgemini** is using Google Cloud to build AI agents that help optimize the ecommerce experience by helping retailers accept customer orders through new revenue channels and accelerate the order-to-cash process for digital stores. * ***Zippedi**, a Chilean data capture platform, uses Google Cloud’s gen AI to power its robots and deliver real-time insights to its customers. * ***Thales** is developing a global Security Operation Centre platform based on Google Cloud cybersecurity technologies and expertise, including Google Security Operations platform, VirusTotal, and Mandiant Threat Intelligence, all powered by gen AI. * \u200b\u200b***Unico**, a Brazilian technology company that validates people’s real identities to ensure data privacy, puts Google Cloud’s AI technologies at the core of some of its data protection solutions to help manage scale and security.”, ‘This ability raises challenging questions about how developers and users of generative AI can remain compliant with privacy, security, and intellectual property regulations, making the need to establish clear guardrails and guiding ethical principles paramount. Businesses need a clear understanding of how to use generative AI responsibly and how to align their goals for the technology with their company values to protect customers, data, and their business operations, and vendors need legal and ethical frameworks for developing and training GenAI tools to ensure they’re contributing to the appropriate use of the technology moving forward. The core best practices for ethical use of generative AI focus on training employees, implementing data security procedures, continuously fact-checking an AI system’s output, and establishing acceptable use policies.’, ‘What are the primary ethical challenges arising from using generative technologies, specifically concerning privacy, data protection, copyright infringement,’, ‘I had already been thinking about the cultural backlash that Big Generative AI was facing, and something like this opens up even more space for people to be critical of the practices and promises of generative AI companies. From what I can see, tech companies think they can just wear people down, forcing them to accept that generative AI is an inescapable part of all their software now, whether it works or not. “I had already been thinking about the cultural backlash that Big Generative AI was facing, and something like this opens up even more space for people to be critical of the practices and promises of generative AI companies.’, ‘The pro-AI counter-argument here is that when GPT-4o learns from a copyrighted book, it’s the same as when a human learns from it. It’s an argument that training AI is unfair in some other way. The second category of anti-AI argument is that it’s wrong because it’s harmful to the people who use it. The third category of anti-AI argument is that it’s bad for the environment. For the sake of completeness, I want to acknowledge but not discuss three anti-AI arguments, which I think are either obviously wrong or not coming from the left-wing anti-AI backlash that I’m discussing in this post. That *is* an argument some left-wing anti-AI people make, but I think it’s simply wrong.’]}}
{‘generate’: {‘draft’: “**Title: Understanding the Divide: AI vs. Generative AI**\n\nI. Introduction\nArtificial Intelligence (AI) has revolutionized technology, but within this realm lies Generative AI, a subset that takes innovation to new heights. While AI focuses on data analysis and automation, Generative AI delves into creating original content. This essay delves into the distinctions between AI and Generative AI, exploring their definitions and applications.\n\nII. Understanding Artificial Intelligence (AI)\nAI refers to the simulation of human intelligence processes by machines, including learning, reasoning, and problem-solving. It excels in data analysis and prediction, enhancing various sectors like healthcare, finance, and transportation through applications such as predictive analytics and virtual assistants.\n\nIII. Exploring Generative AI\nGenerative AI, a subset of AI, is designed to produce new content like text and images based on patterns learned from data. Unlike traditional AI, Generative AI is not confined to specific tasks, allowing for interactive and collaborative content creation. Its applications range from personalized content generation to creative storytelling.\n\nIV. Key Differences Between AI and Generative AI\n1. Data Processing Capabilities: AI analyzes existing data, while Generative AI generates new content based on learned patterns.\n2. Creativity and Innovation: Generative AI adds a creative twist by producing original content, unlike AI that focuses on data interpretation.\n3. Learning and Adaptation: Generative AI learns from data to create novel outputs, showcasing adaptability and innovation beyond traditional AI’s scope.\n\nV. Real-World Examples\nAI applications span across industries like customer support, healthcare, and finance, streamlining operations and enhancing user experiences. Generative AI finds its place in platforms like Capgemini, Zippedi, and Thales, optimizing ecommerce, data capture, and security operations through innovative content creation.\n\nVI. Future Implications\nAs AI and Generative AI advance, ethical considerations become paramount. Businesses must align technology use with ethical principles to safeguard privacy and data integrity. The future holds promise for AI and Generative AI, but responsible development and usage are crucial for sustainable innovation.\n\nVII. Conclusion\nIn conclusion, AI and Generative AI represent distinct facets of artificial intelligence, with AI focusing on data analysis and prediction, while Generative AI pioneers in creative content generation. Understanding these differences and their applications is vital for navigating the evolving landscape of AI technologies responsibly and ethically.”, ‘revision_number’: 3}}
```


The output stream shows the agent’s thought process step-by-step:

1. Planner: It first creates a detailed outline for the essay.
2. Research Plan: It generates and executes search queries to gather initial content.
3. Generate: It writes the first draft based on the plan and research.
4. Reflect: The agent critiques its own draft, suggesting deeper analysis and real-world examples.
5. Research Critique: Based on the critique, it generates new queries to find more specific information.
6. Generate: Finally, it writes a revised, improved draft, incorporating the feedback and new research.

---

In this final article, we’ve built a powerful, multi-agent essay writer that demonstrates the core strengths of LangGraph: managing complex states, orchestrating multiple specialized agents, and creating cyclical, reflective workflows.

By breaking down a complex task like essay writing into smaller, manageable roles — planning, researching, writing, and critiquing — we created an agent that can iteratively improve its own work.

This concludes our series on building agents with LangGraph. You now have the foundational knowledge to create your own sophisticated, stateful, and controllable AI agents for a wide range of applications. Happy building!
